Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* No ResNet
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 80 blocks (MaxPool2D in each second block)

In [ ]:
out_channels = 8
size = 64

model_blocks = [
    nn.Conv2d(3, out_channels, 3, 1, 1),
    nn.BatchNorm2d(out_channels),
    nn.ReLU()
]

for stage in range(5):

    for i in range(20):

        conv = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        bn = nn.BatchNorm2d(out_channels)

        model_blocks.extend([conv, bn, nn.ReLU()])

    if stage < 3:
        model_blocks.append(nn.MaxPool2d(2, 2))
        size //= 2
    if stage<4:
        model_blocks.extend([
            nn.Conv2d(out_channels, out_channels*2, 3, 1, 1),
            nn.BatchNorm2d(out_channels*2),
            nn.ReLU()
    ])

        out_channels *= 2

print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(out_channels * size * size, out_channels),
    nn.ReLU(),
    nn.Linear(out_channels, 3)
    )

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment6/",
    save_checkpoints=10,
    print_every=5
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment6/tables/training_metrics.csv", index=False)

### Training/Validation Trend (100 epochs)
* The model starts with very low performance, with training and validation accuracy around 34%, indicating near-random initial classification.
* Training loss gradually decreases from 1.0986 to around 1.07, showing slow optimization progress.
* Validation loss does not consistently decrease and fluctuates between approximately 1.09 and 1.23, indicating unstable generalization.
* Training accuracy improves slowly from 34.36% to a maximum around 39–40%, showing limited learning capacity.
* Validation accuracy remains mostly between 34% and 40%, with only occasional improvements.
* Training precision increases over epochs, reaching values above 0.5 in some epochs, but this improvement is inconsistent.
* Training recall remains relatively stable around 35–38%, showing that the model is not significantly improving its ability to identify all classes.
* Training F1-score improves from approximately 0.18 to around 0.32, but remains low overall.
* Validation precision fluctuates heavily, sometimes becoming high due to prediction bias toward specific classes rather than balanced improvement.
* Validation recall remains around 35–40%, indicating limited class recognition ability.
* Validation F1-score remains low throughout training, generally staying between 0.18 and 0.29.
* The confusion matrices show strong class prediction imbalance, with the model frequently predicting one dominant class and struggling with other classes.
* Around later epochs, the model improves slightly but does not show a clear convergence pattern.
* The best validation performance occurs around epoch 83, where the model achieves its highest validation F1-score and accuracy.
* The model does not show strong overfitting because training performance itself remains low; instead, it appears to suffer from underfitting or insufficient feature learning.

Overall, the model learns gradually but achieves only limited improvement across 100 epochs. Training loss decreases slightly, but validation performance remains unstable, suggesting that the learned representations are not sufficiently discriminative. The model shows a tendency toward biased class predictions, as visible from the confusion matrices where some classes are frequently misclassified. Although later epochs provide small improvements, the overall accuracy and F1-score remain low, indicating that the current architecture, training strategy, or data representation may need further improvement.

<b>Best Epoch 83</b>

<b>Loss</b>
* Train Loss = 1.0836250540
* Valid Loss = 1.1683253749

<b>Training Metrics</b>
* Train Accuracy = 0.3738185167
* Train Precison = 0.3970238268
* Train Recall = 0.3662246764
* Train F1 = 0.2891205251

<b>Validation Accuracy</b>
* Validation Accuracy = 0.4070796371
* Validation Precision = 0.3504330218
* Validation Recall = 0.3987066746
* Validation F1 = 0.2978943586

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

tester = Tester(
    model,
    test_loader,
    3,
    torch.nn.CrossEntropyLoss(),
    "cuda"
)

test_scores = tester.test_all_checkpoints(
    "../models/experiment6"
)

### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment6/tables/test_metrics.csv", index=False)

### Test Performance Trend
* The model begins with weak testing performance, with accuracy around 34.6%, indicating poor initial generalization.
* Test loss decreases from 1.28 in early epochs to around 1.04–1.06 in later epochs, showing some optimization improvement.
* Test accuracy gradually improves from 34.6% to a peak of 42.5% at epoch 83.
* Precision fluctuates significantly across epochs, with occasional high values caused by uneven class predictions rather than consistent improvement.
* Recall remains relatively stable around 33–41%, indicating limited improvement in detecting all classes.
* F1-score improves slowly from 0.17 initially to a maximum of approximately 0.315 at epoch 83.
* The best testing performance occurs around epoch 83, where accuracy, precision, recall, and F1-score are simultaneously highest.
* After epoch 83, testing performance declines, especially by epoch 100 where accuracy drops to 35.1% and F1-score falls to 0.185.
* The model shows unstable generalization, with several epochs producing higher precision but lower recall, suggesting prediction bias toward certain classes.
* The testing behaviour closely matches the validation behaviour, indicating that the validation results are representative of real model performance.
* The model does not achieve strong classification performance, suggesting that the learned features are insufficient for separating the classes effectively.

The model shows unstable learning behavior with an early drop in loss followed by long periods of stagnation in performance, where accuracy remains mostly in the 0.35–0.40 range and loss oscillates around ~1.05–1.09. Precision and recall vary significantly across epochs, indicating inconsistent class-level predictions and poor convergence stability. Although there are occasional spikes in performance, they are not sustained until a clear peak at epoch 83, after which the model again declines, suggesting the training process does not fully converge and is sensitive to optimization noise.